# Apache Avro - Python

All 9 Python examples from [docs/avro.md](https://platob.github.io/yggdryl/avro/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

## Arrow batch reads and writes

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
batch = lambda ids, venues: pa.record_batch(
    {"id": ids, "venue": venues}, schema=schema
)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.avro")
handle.overwrite_arrow_batch(batch([1, 2], ["XNAS", "XNYS"]))
handle.append_arrow_batch(batch([3], ["XLON"]))

merging = handle.record_options()
merging.merge_by_names = ["id"]
handle.merge_arrow_batch(batch([2, 4], ["XPAR", None]), options=merging)

assert handle.read_arrow_reader().read_all().num_rows == 4

### Block encoding options

In [ ]:
from yggdryl import RecordOptions

options = RecordOptions("trades.avro")
assert options.block_codec == "deflate"
assert options.sync_marker is None

options.block_codec = "zstandard"
options.sync_marker = b"0123456789abcdef"
assert options.sync_marker == b"0123456789abcdef"

## Flexible Scalar containers and schema methods

In [ ]:
from yggdryl import avro

schema = {
    "type": "record",
    "name": "trade",
    "fields": [
        {"name": "symbol", "type": "string"},
        {"name": "quantity", "type": "long"},
    ],
}
encoded = avro.dumps(
    [{"symbol": "AAPL", "quantity": 100}, {"symbol": "MSFT", "quantity": 25}],
    schema,
    metadata={"source": "docs"},
)
decoded = avro.loads(encoded)

assert decoded.metadata == {"source": "docs"}
assert decoded.rows[0] == {"quantity": 100, "symbol": "AAPL"}

## Schemas, canonical form, and fingerprints

In [ ]:
from yggdryl import avro

document = {
    "type": "record",
    "name": "trade",
    "doc": "one fill",
    "fields": [
        {"name": "symbol", "type": "string"},
        {"name": "qty", "type": "long", "field-id": 2},
    ],
}
schema = avro.Schema(document)

assert "doc" not in schema.into_canonical_form()
assert schema.fingerprint().to_bytes(8, "little")[0] == 0xF5
assert schema.into_json()["fields"][1]["field-id"] == 2

## Logical types decode as what they mean

In [ ]:
from datetime import date, datetime, timezone
from decimal import Decimal

from yggdryl import avro

schema = {
    "type": "record",
    "name": "row",
    "fields": [
        {"name": "day", "type": {"type": "int", "logicalType": "date"}},
        {"name": "at", "type": {"type": "long", "logicalType": "timestamp-micros"}},
        {"name": "price", "type": {"type": "bytes", "logicalType": "decimal",
                                    "precision": 10, "scale": 2}},
    ],
}
row = {
    "day": date(2024, 2, 29),
    "at": datetime(2023, 11, 14, 22, 13, 20, tzinfo=timezone.utc),
    "price": Decimal("187.50"),
}

decoded = avro.loads(avro.dumps([row], schema)).rows[0]
assert decoded == row

## Reading with a different schema

In [ ]:
from yggdryl import avro

writer = {
    "type": "record",
    "name": "trade",
    "fields": [
        {"name": "symbol", "type": "string"},
        {"name": "qty", "type": "int"},
        {"name": "venue", "type": "string"},
    ],
}
reader = avro.Schema({
    "type": "record",
    "name": "trade",
    "fields": [
        {"name": "quantity", "aliases": ["qty"], "type": "long"},
        {"name": "note", "type": "string", "default": "none"},
    ],
})
encoded = avro.dumps([{"symbol": "AAPL", "qty": 100, "venue": "XNAS"}], writer)

assert avro.loads(encoded, reader_schema=reader).rows == [
    {"note": "none", "quantity": 100}
]

## Streaming a large container

In [ ]:
from yggdryl import avro

schema = {
    "type": "record",
    "name": "row",
    "fields": [{"name": "id", "type": "long"}],
}
stream = avro.blocks(avro.dumps([{"id": 1}, {"id": 2}, {"id": 3}], schema))

assert stream.schema.kind == "record"
block = next(stream)
assert block.count == len(block.rows())

## Single-object encoding

In [ ]:
from yggdryl import avro

schema = avro.Schema({
    "type": "record",
    "name": "tick",
    "fields": [{"name": "price", "type": "double"}],
})
framed = avro.dumps_single({"price": 187.5}, schema)

assert framed[:2] == b"\xc3\x01"
assert avro.loads_single(framed, schema) == {"price": 187.5}

## Codecs and limits

In [ ]:
from yggdryl import avro

encoded = avro.dumps([7], '"long"')
decoded = avro.loads(
    encoded,
    max_depth=8,
    max_input_bytes=1_024,
    max_nodes=8,
)

assert decoded.rows == [7]